# Host window (DMA) — a core writes results straight into PS DDR4, the host reads them back

The smallest end-to-end test of the **host window** (spec
[22-host-window](../specs/software/22-host-window.md)): an on-core `@kernel` stores into an
`Array(host=True)`, the funnel forwards every store over `S_AXI_HP0_FPD` into a CMA buffer in the
PS DDR4, and `rq.run` reads it back with `read_host` — no core RAM involved. Run this before any
experiment that captures more than the core's RAM holds (the raw-IQ `ReadoutCalibration`, or a
`Resonator` scan keeping every shot).

Hardware notebook (`RemoteDriver`, not executed in CI), same shape as
[`remote_pulse.ipynb`](remote_pulse.ipynb). The bundle must come from a build **with** the window
(`riscq.soc` at or after spec 22 W1), e.g. `build/x6y3-hostwin` — an older bitstream has no funnel and
the window arrays read back as zeros.

In [ ]:
import numpy as np

from riscq import run as rq
from riscq.driver.remote import RemoteDriver, upload_bundle
from riscq.lang import Array, compile_kernel, kernel
from riscq.map import SocMap, SocParams

BOARD = "192.168.1.122"                   # the ZCU216's LAN address (or the full PYRO: uri)

drv = RemoteDriver(BOARD, 9091)
print("server:", drv.board.info())

## The bundle

First time only: push the window-capable x6y3 build up and load it. On `load` the board driver
allocates the result buffer (`qubit_num << hostwin_bits` bytes) from the kernel's CMA pool and refuses
loudly if `CmaTotal`/`CmaFree` cannot hold it. The stock ZCU216 PYNQ image has a 128 MB pool, which
is why the x6y3 build sets `"hostwin_bits": 23` (8 MB per core, 64 MB total); the alternative is a
larger `cma=` boot argument.

In [ ]:
print("bundles on the board:", drv.board.bundles())

# upload_bundle(drv, "x6y3-hostwin",
#               xsa="../build/x6y3-hostwin/PulseTableSoc.xsa",           # write_hw_platform export
#               params_json="../software/configs/x6y3.json")             # the SAME JSON the build used
# info = drv.board.load("x6y3-hostwin")                                  # RF bring-up + CMA buffer
# assert info["mts_result"] == 0, "multi-tile sync missed its target latencies"

m = SocMap(SocParams.from_json(drv.board.get_params()))
print(f"connected to '{m.params.name}': {m.params.qubit_num} cores, "
      f"{m.mem_bytes // 1024} KB core RAM, {m.HOSTWIN_BYTES >> 20} MB window per core")

## The kernel

One loop stores the same value into a RAM array and a window array. The kernel body is identical —
`out[i] = ...` is a plain store either way — only the binding differs: `Array(N)` is a `.bss` object in
the core's RAM, `Array(N, host=True)` is a pointer into the write-only window, so it has no RAM
object and no ELF symbol (`prog.host_arrays` carries its window offset instead).

In [ ]:
N = 16


@kernel
def k_window(ram: Array, win: Array, offset: int, n: int):
    for i in range(n):
        ram[i] = i + offset
        win[i] = i + offset


prog = compile_kernel(k_window, m, ram=Array(N), win=Array(N, host=True))
print("window arrays (name: (byte offset, words)):", prog.host_arrays)
print("RAM symbol for 'ram':", hex(prog.var_addr("ram")), "— 'win' has none (it lives in DDR4)")

## Run it — the window array must equal the RAM array

`rq.run` = `setup` (points the funnel at this driver's CMA buffer while the reset is held, loads the
image) + one `rerun` (params in → run → poll DONE → results out). A window result comes back through
`drv.read_host`, a RAM result through the bus, and `rq.run` picks per name. Runs on core 0; every other
core is parked.

In [ ]:
out = rq.run(drv, m, {0: prog}, params={0: dict(offset=100, n=N)}, results=["ram", "win"],
             timeout=5_000)[0]
print("ram:", list(out["ram"]))
print("win:", list(out["win"]))
assert list(out["win"]) == list(out["ram"]) == [i + 100 for i in range(N)], "host window mismatch"

# a second rerun overwrites the window in place (it is not RAM, so nothing re-zeroes it)
out = rq.rerun(drv, m, {0: prog}, params={0: dict(offset=5000, n=N)}, results=["win"],
               timeout=5_000)[0]
assert list(out["win"]) == [i + 5000 for i in range(N)], f"stale window data: {list(out['win'])}"
print("second run OK — the window is rewritten in place")

## Beyond the core RAM

The point of the window: a result larger than the whole core RAM. The same array without
`host=True` would not even link. Here 4× the RAM (in words) — pick any size up to the 16 MB slice.

In [ ]:
BIG = m.mem_bytes                          # words = 4x the core RAM in bytes


@kernel
def k_big(win: Array, offset: int, n: int):
    for i in range(n):
        win[i] = i + offset


big = compile_kernel(k_big, m, win=Array(BIG, host=True))
out = rq.run(drv, m, {0: big}, params={0: dict(offset=7, n=BIG)}, results=["win"],
             timeout=5_000)[0]
win = np.asarray(out["win"])
assert np.array_equal(win, np.arange(BIG) + 7), "big window array mismatch"
print(f"{BIG} words ({4 * BIG >> 10} KB) through the window OK")

drv.close()
print("done")